# 05 - Data augmentation and honest model evaluation

**Goal:** compare two controlled training pipelines, select using validation data, and
explain final test errors with a confusion matrix, per-class metrics, and an image gallery.

**Before you start:** lessons 02-04's CNNs, train/eval modes, and validation checkpoints.
No saved model is needed; both candidates are trained here from the same starting weights.
Use **Restart Kernel and Run All Cells** for a clean experiment.

**Live demonstration:** 60-90 minutes including interpretation. Two small training runs
take longer than one; actual runtime varies. `QUICK=True` limits the dataset and epochs.
Smoke mode verifies code paths; its tiny samples cannot provide reliable model-quality estimates.

**Predict first:** will small rotations and shifts help on this dataset? We will measure
the result rather than assume augmentation must improve accuracy.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## 1. Fix the comparison before training

Both candidates use the same examples, split, architecture, initialization, batch size,
optimizer settings, and epoch budget. They differ in training transforms.
The selection rule is **lowest validation loss**. Decide this before seeing test results.


In [ ]:
from copy import deepcopy
import pandas as pd
from IPython.display import display
from torchvision import transforms
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score)

SEED = 42
cu.seed_everything(SEED)
TRAIN_N = 192 if SMOKE else (6000 if QUICK else 48000)
VAL_N = 48 if SMOKE else (800 if QUICK else 12000)
TEST_N = 48 if SMOKE else (1000 if QUICK else 10000)
EPOCHS = 1 if SMOKE else (5 if QUICK else 15)
BATCH_SIZE = 64
MEAN, STD = (0.5,), (0.5,)
print({"train": TRAIN_N, "validation": VAL_N, "test": TEST_N,
       "epochs_per_candidate": EPOCHS, "smoke": SMOKE})


## 2. Define training and evaluation transforms separately

Small affine transforms rotate, translate, and resize clothing images while usually
preserving their labels. Label preservation is a task-specific assumption to inspect.
We avoid vertical flips and aggressive crops that could destroy useful information.

Only augmented **training** examples receive randomness. Validation and test use the same
deterministic conversion and normalization. This keeps the evaluation question stable.
Black padding is added before normalization, so background remains background after scaling.


In [ ]:
eval_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
augmented_transform = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.95, 1.05), fill=0),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
print("Evaluation / baseline training:", eval_transform)
print("Augmented training:", augmented_transform)


## 3. Make identical splits with independent transforms

A common mistake is to split one dataset and then change its shared `transform`, accidentally
augmenting validation images too. The course helper creates independent dataset views for
training and evaluation. Reusing the seed and limits gives both candidates the same indices.
The assertions below verify split identity instead of merely assuming it.


In [ ]:
split_options = dict(train_limit=TRAIN_N, val_limit=VAL_N, test_limit=TEST_N, seed=SEED)
train_plain, val_plain, test_plain, class_names = cu.fashion_data(
    train_transform=eval_transform, eval_transform=eval_transform, **split_options,
)
train_aug, val_aug, test_aug, aug_class_names = cu.fashion_data(
    train_transform=augmented_transform, eval_transform=eval_transform, **split_options,
)
assert list(train_plain.indices) == list(train_aug.indices)
assert list(val_plain.indices) == list(val_aug.indices)
assert list(test_plain.indices) == list(test_aug.indices)
assert list(class_names) == list(aug_class_names)
assert torch.equal(val_plain[0][0], val_aug[0][0])
print("Verified identical example indices and deterministic validation preprocessing.")
val_loader = cu.loader(val_plain, BATCH_SIZE, shuffle=False)


## 4. Inspect augmentation before trusting it

Fetching the same training index repeatedly produces different random views, but its label
stays the same. Inspect whether a person can still recognize the item in each view.
Data augmentation adds variation; it does not add genuinely new labeled observations.

**Audience prompt:** which transforms would be unsuitable for digits, medical scans,
or traffic signs? Explain the label-preservation assumption for each task.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
original, actual = train_plain[0]
axes.flat[0].imshow((original[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
axes.flat[0].set_title(f"Original: {class_names[actual]}", fontsize=9)
for index in range(1, 8):
    changed, label = train_aug[0]
    assert label == actual
    axes.flat[index].imshow((changed[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    axes.flat[index].set_title(f"Random view {index}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Store a common initialization

`SmallCNN` is the reusable convolutional model defined in `course_utils.py`.
Print its structure so the comparison is explicit. Copy one initial state and load it into
each candidate. This avoids attributing an initialization difference to augmentation.

The loader uses its own seeded random generator, so batch order is controlled independently
of random augmentation. This model has no dropout. Seeds improve repeatability, but exact
equality across hardware is not promised. Repeated independent seeds would provide stronger
evidence than this single-run teaching comparison.


In [ ]:
cu.seed_everything(SEED)
template_model = cu.SmallCNN(num_classes=10, in_channels=1)
initial_state = deepcopy(template_model.state_dict())
print(template_model)
print("Parameters:", sum(p.numel() for p in template_model.parameters()))
del template_model


## 6. Train both candidates and retain their best validation checkpoints

Each run resets the seed and builds a new shuffled loader with the same seed.
`cu.fit` restores its lowest-validation-loss checkpoint. We store CPU copies so there is
only one actively trained model on the GPU at a time.
No test results are calculated during candidate selection.


In [ ]:
histories, saved_states, comparisons = {}, {}, []
for name, training_dataset in [("baseline", train_plain), ("augmented", train_aug)]:
    print(f"\nTraining candidate: {name}")
    cu.seed_everything(SEED)
    candidate = cu.SmallCNN(num_classes=10, in_channels=1).to(DEVICE)
    candidate.load_state_dict(initial_state)
    candidate_loader = cu.loader(training_dataset, BATCH_SIZE, shuffle=True, seed=SEED)
    history = cu.fit(candidate, candidate_loader, val_loader, epochs=EPOCHS,
                     device=DEVICE, lr=1e-3, weight_decay=1e-4)
    selected_metrics = cu.evaluate(candidate, val_loader, DEVICE)
    histories[name] = history
    saved_states[name] = {key: value.detach().cpu().clone()
                          for key, value in candidate.state_dict().items()}
    comparisons.append({"candidate": name,
                        "best_epoch": min(history, key=lambda row: row["val_loss"])["epoch"],
                        "val_loss": selected_metrics["loss"],
                        "val_accuracy": selected_metrics["accuracy"]})
    del candidate, candidate_loader
comparison_table = pd.DataFrame(comparisons).set_index("candidate")
display(comparison_table)


## 7. Compare validation curves and choose the winner

Augmentation can make training harder and may need more epochs to help. On a short run,
it may underperform the baseline. Report what happened rather than changing the story.
Use validation loss for the declared selection rule even if another candidate has slightly
higher accuracy. Loss accounts for the model's scores as well as its top class.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in histories.items():
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["val_loss"] for row in history], marker="o", label=name)
    axes[1].plot(epochs, [row["val_accuracy"] for row in history], marker="o", label=name)
axes[0].set_title("Validation cross-entropy")
axes[1].set_title("Validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

winner = min(comparisons, key=lambda row: row["val_loss"])["candidate"]
selected_model = cu.SmallCNN(num_classes=10, in_channels=1).to(DEVICE)
selected_model.load_state_dict(saved_states[winner])
print("Selected using validation loss:", winner)


## 8. Freeze the choice, then use the test set

Now evaluate only the selected candidate on deterministic test inputs.
We cache logits and labels once and use that same result for all following reports.
Looking at errors is valuable, but changes inspired by these test errors require a fresh
final holdout if you want a new unbiased estimate.

Accuracy is the fraction of correct predictions. Balanced accuracy averages recall over
the true classes present; macro F1 below gives each of the ten requested classes equal
weight. These stratified smoke samples include every class, but per-class counts are tiny,
so the resulting metrics are unstable.


In [ ]:
test_loader = cu.loader(test_plain, BATCH_SIZE, shuffle=False)
test_logits, test_labels = cu.predict(selected_model, test_loader, DEVICE)
y_true = test_labels.numpy()
y_pred = test_logits.argmax(dim=1).numpy()
all_class_ids = list(range(len(class_names)))
summary = {
    "selected_candidate": winner,
    "test_examples": len(y_true),
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_f1_all_10_classes": f1_score(y_true, y_pred, labels=all_class_ids,
                                       average="macro", zero_division=0),
    "cross_entropy": torch.nn.functional.cross_entropy(test_logits, test_labels).item(),
}
display(pd.Series(summary, name="Held-out test assessment"))


## 9. Interpret precision, recall, F1, and support

**Precision:** of the images predicted as a class, what fraction really belongs to it?
**Recall:** of the true examples of a class, what fraction did we find?
**F1:** the harmonic mean of precision and recall. **Support:** the number of true examples.
For one class, precision is `TP/(TP+FP)` and recall is `TP/(TP+FN)`.

Macro averages give each class equal weight; weighted averages weight by support.
We use `zero_division=0` so a class with no predicted examples receives a displayed zero
precision rather than an undefined numerical result. Treat absent-class metrics cautiously.


In [ ]:
report = classification_report(y_true, y_pred, labels=all_class_ids,
                               target_names=list(class_names), output_dict=True,
                               zero_division=0)
report_table = pd.DataFrame(report).T
display(report_table.round(3))


## 10. Read a confusion matrix in the correct direction

**Rows are true classes; columns are predicted classes.** The diagonal contains correct
predictions. A large off-diagonal count identifies a common confusion.
The second matrix normalizes each true-class row, making recall patterns easier to compare
when classes have different supports. A zero-support class has an all-zero normalized row.


In [ ]:
counts = confusion_matrix(y_true, y_pred, labels=all_class_ids)
row_totals = counts.sum(axis=1, keepdims=True)
normalized = np.divide(counts, row_totals, out=np.zeros_like(counts, dtype=float),
                       where=row_totals != 0)
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
ConfusionMatrixDisplay(counts, display_labels=class_names).plot(
    ax=axes[0], xticks_rotation=90, colorbar=False, values_format="d")
ConfusionMatrixDisplay(normalized, display_labels=class_names).plot(
    ax=axes[1], xticks_rotation=90, colorbar=False, values_format=".2f")
axes[0].set_title("Counts: true rows, predicted columns")
axes[1].set_title("Fraction within each true class")
plt.tight_layout()
plt.show()


## 11. Find the largest class confusions

An error table helps turn a large matrix into an investigation plan.
Check whether the images are visually similar, labels are ambiguous, or the model ignores
useful features. A confusion count alone does not reveal the cause.


In [ ]:
confusion_pairs = [
    {"true_class": class_names[true_id], "predicted_class": class_names[pred_id],
     "errors": int(counts[true_id, pred_id])}
    for true_id in all_class_ids for pred_id in all_class_ids
    if true_id != pred_id and counts[true_id, pred_id] > 0
]
if confusion_pairs:
    display(pd.DataFrame(confusion_pairs).sort_values("errors", ascending=False).head(8))
else:
    print("No mistakes in this evaluated sample; inspect a larger independent sample.")


## 12. Inspect confident mistakes, not just averages

Sort incorrect predictions by their maximum softmax score. A wrong answer with a large
score is a useful reminder that softmax is not a guarantee of correctness.
The gallery follows the same unshuffled test order used to produce the cached predictions.
Later lessons examine interpretation and calibration more directly.


In [ ]:
scores = test_logits.softmax(dim=1).max(dim=1).values.numpy()
wrong_indices = np.flatnonzero(y_pred != y_true)
ordered_errors = wrong_indices[np.argsort(-scores[wrong_indices])]
if len(ordered_errors) == 0:
    print("No errors found in this sample.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, index in zip(axes.flat, ordered_errors[:8]):
        image, actual = test_plain[int(index)]
        ax.imshow((image[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"True: {class_names[actual]}\n"
                     f"Pred: {class_names[y_pred[index]]}\nScore: {scores[index]:.2f}",
                     fontsize=9, color="crimson")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 13. Save the experiment evidence

Record the validation comparison separately from the final test report.
Keep the selected model's weights, class order, and preprocessing with the result.
These named files are replaced when you rerun this lesson. Smoke results are labeled in
the checkpoint and summary so they are not mistaken for real training evidence.


In [ ]:
import json

comparison_table.to_csv(cu.ARTIFACTS / "05_validation_comparison.csv")
report_table.to_csv(cu.ARTIFACTS / "05_test_classification_report.csv")
pd.DataFrame(counts, index=class_names, columns=class_names).to_csv(
    cu.ARTIFACTS / "05_test_confusion_matrix.csv")
summary["smoke"] = bool(SMOKE)
summary["quick"] = bool(QUICK)
(cu.ARTIFACTS / "05_evaluation_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8")
torch.save({"model_state": saved_states[winner], "architecture": "SmallCNN",
            "in_channels": 1, "num_classes": 10, "class_names": list(class_names),
            "mean": list(MEAN), "std": list(STD), "image_size": [28, 28],
            "selected_candidate": winner, "seed": SEED, "smoke": SMOKE},
           cu.ARTIFACTS / "05_selected_cnn.pt")
print("Saved comparison, test metrics, confusion matrix, summary, and selected weights to:")
print(cu.ARTIFACTS)


## 14. Exercises and critical discussion

1. Recalculate precision and recall manually for one class using the confusion matrix.
2. Compare rotation limits `0`, `10`, and `25` degrees using validation data and equal budgets.
   Inspect augmented images before training. Why might a stronger transform hurt?
3. Repeat the baseline/augmented comparison with seeds 7, 42, and 123. Report each validation
   result plus its mean and spread. Do the conclusions survive different seeds?
4. Imagine 95% of images are shirts. Explain why accuracy alone could be misleading.

<details><summary>Solution hints</summary>

For class `k`, true positives are `counts[k,k]`; predicted positives are column `k`'s sum;
actual positives are row `k`'s sum. Large rotations can introduce images that do not resemble
realistic examples, or remove distinguishing content. A classifier that always guesses
shirt could have 95% accuracy in the imagined dataset while finding none of the other classes.

```python
k = 0
true_positive = counts[k, k]
precision = true_positive / counts[:, k].sum() if counts[:, k].sum() else 0.0
recall = true_positive / counts[k, :].sum() if counts[k, :].sum() else 0.0
print(class_names[k], {"precision": precision, "recall": recall})
```

</details>

**Presenter checkpoint:** ask a learner to point to the validation-only selection line and
explain why both candidates were not selected by their test scores. Then ask them to interpret
one off-diagonal matrix entry and one confident error.

**Limitations:** one split and one seed are weak evidence of a general improvement. Dataset
labels may contain ambiguity, the test subset is finite, and real images can differ from
FashionMNIST. Do not infer deployment readiness from these demonstrations.

**References:** [Torchvision transforms](https://docs.pytorch.org/vision/stable/transforms.html),
[scikit-learn classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics),
[confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).
